# DRAGNET — materialize per-case rows on existing cells (extraction + orders)

The first study commits printed the tables but did not persist per-case rows. This notebook
re-runs, on the downloaded cells, exactly what those tables need — games are deterministic
(greedy decode, fixed seed), so the numbers reproduce:

- **AND cells → `extraction.jsonl`** (per case × arm): the paired arm-vs-arm test, H4.
- **natural cells → `orders.jsonl`** (DRAGNET interaction + shapley rankings): lets the conformal
  guarantee (H2) be scored against the coalition-aware order, not just the ContextCite baseline.

**Before running:** Settings → Accelerator **GPU T4 x2** · Internet **on** · **Add Input** →
attach the dataset holding your downloaded `scope_*` results. Rough budget: ~1–2 h per 7B cell.

In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working')
os.chdir(WORK)

def fetch(name, url):
    if (WORK / name).exists():
        subprocess.run(['git', '-C', name, 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', url, name], check=True)

fetch('lineup', 'https://github.com/santoshcheethiralame-dot/LINEUP')
fetch('dragnet', 'https://github.com/santoshcheethiralame-dot/DRAGNET')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', './lineup[gpu]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', './dragnet'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'bitsandbytes>=0.46.1'], check=True)
import bitsandbytes
print('[ok] lineup + dragnet installed; bitsandbytes', bitsandbytes.__version__)

In [ ]:
CONDITIONS = ('and',)     # AND cells for extraction; add 'hardtraps' for the OR cells too
EXTRACT_LIMIT = 40        # wrong cases per cell; a superset of the original run's cases
RUN_EXTRACTION = True     # the H4 rows on the CONDITIONS cells
RUN_ORDERS = True         # log DRAGNET's interaction/shapley orders on the natural cells (H2)
ORDERS_LIMIT = 0          # wrong cases per natural cell for orders (0 = all)
DEEP_MSCS = False         # re-enumerate the natural cells' minimal sufficient sets at a deeper
                          # bound: the size-3 families cap conformal coverage at ~0.88, below the
                          # alpha=0.1 target -- max_size 5 lifts the ceiling (63 vs 42 queries/case at k=6)
MSCS_MAX_SIZE = 5
MSCS_LIMIT = 100          # wrong cases per natural cell for the deep pass; 100 matches the
                          # original enumeration's case set exactly (deterministic order). 0 = all.
N_SAMPLES = 48
SEED = 0
VALIDATE = False          # True also re-runs validate_designed for per-case validation.jsonl rows
VALIDATE_MAX_SIZE = 3

MODEL_ZOO = {
    'qwen':    'Qwen/Qwen2.5-7B-Instruct',
    'phi':     'microsoft/Phi-3.5-mini-instruct',
    'mistral': 'mistralai/Mistral-7B-Instruct-v0.3',
}

In [ ]:
# Stage every attached run cell into runs/<dataset>/<condition>/<tag>, robust to how the dataset
# was built: Kaggle may auto-extract uploaded archives or keep them zipped, and a zip may or may
# not carry a leading runs/ folder. Find every scenarios.jsonl and key off its last three path
# parts, so all of those layouts land in the same place.
import shutil
import zipfile

scratch = Path('_incoming')
for path in sorted(set(Path('/kaggle/input').glob('**/scope_*.zip')) | set(Path('/kaggle/input').glob('**/dragnet_*.zip'))):
    with zipfile.ZipFile(path) as archive:
        archive.extractall(scratch / path.stem)
    print('unzipped', path.name)

sources = set(Path('/kaggle/input').glob('*/**/scenarios.jsonl')) | set(scratch.glob('**/scenarios.jsonl'))
for marker in sorted(sources):
    parent = marker.parent                       # .../<dataset>/<condition>/<tag>
    if parent.name not in MODEL_ZOO:             # skip lineup/app/sample and other stray jsonl
        continue
    dest = Path('runs', *parent.parts[-3:])
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(parent, dest)
    print('staged', dest)

cells = sorted(
    cell for cond in CONDITIONS for cell in Path('runs').glob(f'*/{cond}/*')
    if (cell / 'scenarios.jsonl').exists() and cell.name in MODEL_ZOO
)
natural = sorted(str(c) for c in Path('runs').glob('*/natural/*') if (c / 'scenarios.jsonl').exists())
if RUN_EXTRACTION:
    assert cells, 'no matching CONDITIONS cells -- check the dataset (or set RUN_EXTRACTION=False)'
print('CONDITIONS cells:', [str(c) for c in cells])
print('natural cells   :', natural)

In [ ]:
if RUN_EXTRACTION:
    for cell in cells:
        model = MODEL_ZOO[cell.name]
        print(f'==== extraction: {cell}  ({model}) ====', flush=True)
        subprocess.run(
            [sys.executable, 'dragnet/scripts/run_extraction.py', '--cell', str(cell),
             '--model', model, '--load-in-4bit', '--n-samples', str(N_SAMPLES),
             '--seed', str(SEED), '--limit', str(EXTRACT_LIMIT)],
            check=True,
        )
        assert (cell / 'extraction.jsonl').stat().st_size > 0
        if VALIDATE:
            print(f'==== validation: {cell} ====', flush=True)
            subprocess.run(
                [sys.executable, 'dragnet/scripts/validate_designed.py', '--cell', str(cell),
                 '--model', model, '--load-in-4bit', '--max-size', str(VALIDATE_MAX_SIZE)],
                check=True,
            )
            assert (cell / 'validation.jsonl').stat().st_size > 0
        print(f'[ok] {cell}', flush=True)

In [ ]:
# Natural-cell passes: DRAGNET's own rankings -> orders.jsonl (the real H2), and optionally the
# deeper minimal-sufficient enumeration -> mscs.jsonl at MSCS_MAX_SIZE (the coverage-ceiling lift).
if RUN_ORDERS or DEEP_MSCS:
    natural_cells = sorted(
        cell for cell in Path('runs').glob('*/natural/*')
        if (cell / 'scenarios.jsonl').exists() and cell.name in MODEL_ZOO
    )
    print('natural cells:', [str(c) for c in natural_cells])
    for cell in natural_cells:
        model = MODEL_ZOO[cell.name]
        if RUN_ORDERS and not (cell / 'orders.jsonl').exists():
            print(f'==== orders: {cell}  ({model}) ====', flush=True)
            args = [sys.executable, 'dragnet/scripts/run_orders.py', '--cell', str(cell),
                    '--model', model, '--load-in-4bit', '--n-samples', str(N_SAMPLES), '--seed', str(SEED)]
            if ORDERS_LIMIT:
                args += ['--limit', str(ORDERS_LIMIT)]
            subprocess.run(args, check=True)
            assert (cell / 'orders.jsonl').stat().st_size > 0
            print(f'[ok] orders {cell}', flush=True)
        if DEEP_MSCS:
            print(f'==== deep mscs (max_size {MSCS_MAX_SIZE}): {cell}  ({model}) ====', flush=True)
            args = [sys.executable, 'dragnet/scripts/run_natural_mscs.py', '--cell', str(cell),
                    '--model', model, '--load-in-4bit', '--max-size', str(MSCS_MAX_SIZE)]
            if MSCS_LIMIT:
                args += ['--limit', str(MSCS_LIMIT)]
            subprocess.run(args, check=True)
            print(f'[ok] deep mscs {cell}', flush=True)

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/dragnet_extraction_rows', 'zip', 'runs')
print('download dragnet_extraction_rows.zip from the Output panel')